# External validation of HSPC motif networks

This notebook packages three existing analyses: STRING v11.5, Blood ChIP-Atlas CoLo, and BioID/AP-MS. It starts from prepared motif networks and external reference files; it does not download data or reconstruct upstream networks.

Four-lineage classes use the top 5% of undirected scores per lineage: common = 4/4, specific = 1/4. CoLo and mass spectrometry also retain 1%, 2%, and 10% sensitivity analyses and the Spearman baseline. The prepared undirected network score is the sum of the absolute reciprocal directed weights; missing lineage pairs are assigned zero before ranking. This is distinct from the 14-cell-type common/specific definition in notebook 17.

Evidence is mapped from TF pairs to motif-cluster pairs through any supporting non-self member pair. STRING measures functional association, CoLo genomic co-localization, BioID proximity, and AP-MS co-complex evidence. They do not establish direct physical binding of each predicted TF pair or locus-specific regulation.

**Historical statistical caveat:** STRING uses selected versus unselected pairs. The original CoLo and BioID/AP-MS scripts compare selected pairs against the *entire* testable universe, including those selected pairs. Their `fisher_or` and `fisher_p_greater` outputs are retained only for historical reproduction and are not a valid independent selected-versus-unselected contrast. Support counts and Wilson 95% binomial intervals are unaffected. No corrected inferential result is substituted here. Random pair sampling in STRING is not degree-matched, and no multiplicity adjustment is applied across these benchmark comparisons.

Dependencies: Python, NumPy, pandas, SciPy, statsmodels, openpyxl. Configure paths, then run sections in order. Outputs are written only to `OUTPUT_ROOT`; reruns replace outputs with the same names.

## Inputs

| Input | Required structure |
| --- | --- |
| Motif mapping CSV | `motif`, `tf`; many-to-many mapping |
| Four-lineage CSV | `motif1`, `motif2`, `edge_class_top5pct` |
| Undirected long CSV | `motif1`, `motif2`, `cell_type`, `score`; 21,120 unique pairs |
| Gene-by-motif directory | `{lineage}.gene_by_motif.csv` for hsc_mpp, memp, elp, cmp |
| STRING v11.5 directory | `9606.protein.info.v11.5.txt.gz`, `9606.protein.links.v11.5.txt.gz` |
| Blood CoLo directory | Official query TSVs named by TF; `Protein` and numeric CoLo columns |
| Mass spectrometry workbook | `41467_2022_28341_MOESM3_ESM.xlsx`; BioID and AP-MS sheets, header row 10 |

CoLo testability follows the source: at least one mapped TF has a downloaded query file. Inspect `chip_atlas_parse_qc.csv`; failed or empty queries must be resolved before interpreting absence of support. Mass spectrometry testability requires at least one mapped bait TF. Each benchmark has its own denominator.

In [ ]:
from pathlib import Path
INPUT_ROOT = Path('/path/to/network_validation_inputs')
OUTPUT_ROOT = Path('results/network_external_validation')
MAPPING = INPUT_ROOT / 'motif_tf.csv'
FOUR = INPUT_ROOT / 'motif_motif_four_lineage_top5pct_wide.csv'
UNDIRECTED = INPUT_ROOT / 'motif_motif_undirected_long.csv'
GENE = INPUT_ROOT / 'gene_by_motif'
STRING_DIR = INPUT_ROOT / 'string_v11_5'
BLOOD_RAW = INPUT_ROOT / 'blood_colo_raw'
MS_XLSX = INPUT_ROOT / '41467_2022_28341_MOESM3_ESM.xlsx'
# Optional: ignored by STRING when the file does not exist.
GLOBAL = INPUT_ROOT / 'global_undirected_lingam_edges.csv'
def require_inputs(paths):
    missing = [str(p) for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Configure the input paths first:\n" + "\n".join(missing))

def prepare_output(path, input_roots):
    path = path.resolve()
    for root in input_roots:
        root = root.resolve()
        if path == root or root in path.parents or path in root.parents:
            raise ValueError("Use an output directory separate from all input directories")
    path.mkdir(parents=True, exist_ok=True)

require_inputs([MAPPING, FOUR, UNDIRECTED, GENE, STRING_DIR, BLOOD_RAW, MS_XLSX])
prepare_output(OUTPUT_ROOT, [INPUT_ROOT])

## STRING v11.5

Strict threshold: `combined_score > 400`. The optional global network is matched to the mapped common-plus-specific set size. Pair sampling uses 10,000 draws; a sorted sampling pool makes seeded runs reproducible across Python processes, so historical Monte Carlo draws may differ.

In [ ]:
#!/usr/bin/env python3
"""Replicate GET's STRING v11.5 motif-cluster benchmark for HSPC networks.

Primary endpoint exactly follows GET: STRING combined_score >400 is mapped from
TF--TF pairs to motif-cluster pairs; precision is positive selected pairs / all
selected pairs.  STRING combined score is an interaction-association benchmark,
not necessarily experimentally demonstrated physical binding.
"""
from pathlib import Path
import gzip, json, logging, random
from collections import defaultdict
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

OUT = OUTPUT_ROOT / 'string_v11_5_benchmark'
SEED = 20260915
N_PERM = 10_000

def read_info(path):
    gene_to_id = {}
    with gzip.open(path, 'rt') as f:
        header = next(f).rstrip('\n').split('\t')
        idx_id, idx_name = header.index('#string_protein_id'), header.index('preferred_name')
        for line in f:
            p = line.rstrip('\n').split('\t')
            if len(p) > max(idx_id, idx_name):
                gene_to_id[p[idx_name].upper()] = p[idx_id]
    return gene_to_id

def read_links(path, allowed):
    hits = set()
    with gzip.open(path, 'rt') as f:
        header = next(f).rstrip('\n').split(' ')
        ia, ib, iscore = header.index('protein1'), header.index('protein2'), header.index('combined_score')
        for line in f:
            p = line.rstrip('\n').split(' ')
            if len(p) <= iscore or int(p[iscore]) <= 400:
                continue
            a, b = p[ia], p[ib]
            if a in allowed and b in allowed:
                hits.add(tuple(sorted((a, b))))
    return hits

def labels_for_universe(motif_to_ids, links):
    motifs = sorted(motif_to_ids)
    labels, rows = {}, []
    for i, a in enumerate(motifs):
        for b in motifs[i+1:]:
            hit_pairs = []
            for ia in motif_to_ids[a]:
                for ib in motif_to_ids[b]:
                    if ia != ib and tuple(sorted((ia, ib))) in links:
                        hit_pairs.append((ia, ib))
            key = (a, b)
            labels[key] = bool(hit_pairs)
            rows.append((a, b, len(motif_to_ids[a]), len(motif_to_ids[b]), int(bool(hit_pairs)), len(hit_pairs)))
    return labels, pd.DataFrame(rows, columns=['motif1','motif2','n_tf_motif1','n_tf_motif2','string_gt_score_gt400','n_supporting_tf_pairs'])

def summarize(name, selected, labels, all_keys, rng):
    selected = sorted(set(selected))
    selected_set = set(selected)
    n, k = len(selected), sum(labels[x] for x in selected)
    universe_k = sum(labels[x] for x in all_keys)
    universe_n = len(all_keys)
    odds, p = fisher_exact([[k, n-k], [universe_k-k, (universe_n-n)-(universe_k-k)]], alternative='greater')
    # Primary GET statistic is precision. Secondary null randomly selects n cluster pairs.
    null = []
    pool = sorted(all_keys)  # Stable ordering across Python processes.
    for _ in range(N_PERM):
        null.append(sum(labels[x] for x in rng.sample(pool, n)) / n)
    observed = k / n
    emp_p = (1 + sum(x >= observed for x in null)) / (N_PERM + 1)
    return {'network':name, 'selected_cluster_pairs':n, 'string_positive_pairs':k,
            'precision':observed, 'universe_cluster_pairs':universe_n,
            'universe_positive_pairs':universe_k, 'universe_precision':universe_k/universe_n,
            'fold_over_universe':observed/(universe_k/universe_n) if universe_k else np.nan,
            'fisher_or':odds, 'fisher_p_one_sided':p,
            'random_mean_precision':float(np.mean(null)), 'random_sd_precision':float(np.std(null, ddof=1)),
            'random_empirical_p_one_sided':emp_p, 'random_permutations':N_PERM}

def main():
    OUT.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(filename=OUT/'run.log', level=logging.INFO, format='%(asctime)s %(message)s')
    for p in (STRING_DIR/'9606.protein.info.v11.5.txt.gz', STRING_DIR/'9606.protein.links.v11.5.txt.gz', FOUR):
        if not p.exists(): raise FileNotFoundError(p)
    mapping = pd.read_csv(MAPPING, encoding='utf-8-sig').drop_duplicates()
    motif_to_genes = mapping.groupby('motif')['tf'].apply(lambda x: sorted(set(str(v).upper() for v in x))).to_dict()
    info = read_info(STRING_DIR/'9606.protein.info.v11.5.txt.gz')
    motif_to_ids = {m: sorted({info[g] for g in genes if g in info}) for m, genes in motif_to_genes.items()}
    motif_to_ids = {m: ids for m, ids in motif_to_ids.items() if ids}
    all_ids = set().union(*map(set, motif_to_ids.values()))
    links = read_links(STRING_DIR/'9606.protein.links.v11.5.txt.gz', all_ids)
    labels, universe = labels_for_universe(motif_to_ids, links)
    universe.to_csv(OUT/'motif_cluster_string_v11_5_ground_truth.csv', index=False)
    all_keys = set(labels)
    four = pd.read_csv(FOUR)
    def key_rows(df): return [tuple(sorted((r.motif1, r.motif2))) for r in df.itertuples() if tuple(sorted((r.motif1, r.motif2))) in all_keys]
    common = key_rows(four.loc[four.edge_class_top5pct.eq('common_4of4')])
    specific = key_rows(four.loc[four.edge_class_top5pct.eq('specific_1of4')])
    top_n = len(common) + len(specific)
    rng = random.Random(SEED)
    results = [summarize('four_lineage_common_4of4', common, labels, all_keys, rng),
               summarize('four_lineage_specific_1of4', specific, labels, all_keys, rng),
               summarize('four_lineage_common_plus_specific', common+specific, labels, all_keys, rng)]
    if GLOBAL.exists():
        global_edges = pd.read_csv(GLOBAL)
        global_selected = key_rows(global_edges.sort_values('rank').head(top_n))
        results.append(summarize(f'GET_style_global_top_{top_n}', global_selected, labels, all_keys, rng))
    pd.DataFrame(results).to_csv(OUT/'string_v11_5_cluster_benchmark_summary.csv', index=False)
    manifest = {'string_release':'v11.5','threshold':'combined_score > 400 (strictly greater, as stated in GET Methods)',
      'primary_endpoint':'precision after TF--TF to motif-cluster mapping','network_inputs':{'four_lineage':str(FOUR),'global':str(GLOBAL)},
      'global_selection':f'top {top_n} global undirected edges, matching common+specific selected-set size; appended when global output exists','random_seed':SEED,
      'caveat':'STRING combined score captures functional association and is not equivalent to experimentally demonstrated direct physical binding.'}
    (OUT/'manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')

if __name__ == '__main__': main()

## Blood ChIP-Atlas CoLo

The primary score cutoff is 4; 9 is a sensitivity cutoff. Take the maximum mapped score across experiments, excluding the `STRING` and `|Average` columns. Fisher columns reproduce the overlapping-background historical calculation described above.

In [ ]:
#!/usr/bin/env python3
"""Direct MS-style external enrichment using official ChIP-Atlas CoLo scores."""
from pathlib import Path
import sys,numpy as np,pandas as pd
from scipy.stats import spearmanr,fisher_exact
from statsmodels.stats.proportion import proportion_confint
klass='Blood'
RAW=BLOOD_RAW
OUT=OUTPUT_ROOT/'blood_colo'
OUT.mkdir(parents=True,exist_ok=True)
LINS=['hsc_mpp','memp','elp','cmp'];COV=[.01,.02,.05,.10]
mp=pd.read_csv(MAPPING,encoding='utf-8-sig').dropna();mp.tf=mp.tf.astype(str).str.upper();mtf=mp.groupby('motif').tf.agg(lambda x:set(x)).to_dict();t2m=mp.groupby('tf').motif.agg(lambda x:set(x)).to_dict()
raw=pd.read_csv(UNDIRECTED);raw['motif_a']=raw[['motif1','motif2']].min(axis=1);raw['motif_b']=raw[['motif1','motif2']].max(axis=1);pairs=sorted(set(zip(raw.motif_a,raw.motif_b)));assert len(pairs)==21120;pidx={x:i for i,x in enumerate(pairs)}
# Parse each official query table. A TF pair score is max across all displayed query/partner experiments.
tfscore={}; query=set();parse=[]
for f in sorted(RAW.glob('*.tsv')):
 q=f.stem.upper();query.add(q)
 try:
  d=pd.read_csv(f,sep='\t'); pcol='Protein';numeric=d.select_dtypes(include='number').columns.tolist();scorecols=[x for x in numeric if x not in []]
  # all numeric columns after metadata are CoLo scores; drop the query average column (not pair score).
  # `STRING` is a database-derived association column in CoLo, not a
  # ChIP-Atlas co-localization score.  Exclude it, as well as the query-TF
  # average column, so this benchmark remains independent of STRING.
  scorecols=[x for x in scorecols if '|Average' not in x and str(x).upper()!='STRING']
  if not scorecols:parse.append({'query_tf':q,'status':'no_score_columns','n_rows':len(d)});continue
  ss=d[scorecols].max(axis=1);n=0
  for partner,v in zip(d[pcol].astype(str).str.upper(),ss):
   if partner in t2m and partner!=q and pd.notna(v):tfscore[(min(q,partner),max(q,partner))]=max(tfscore.get((min(q,partner),max(q,partner)),0),float(v));n+=1
  parse.append({'query_tf':q,'status':'ok','n_rows':len(d),'mapped_partner_rows':n})
 except Exception as e:parse.append({'query_tf':q,'status':'error','error':str(e)})
pd.DataFrame(parse).to_csv(OUT/'chip_atlas_parse_qc.csv',index=False)
test=np.zeros(len(pairs),bool);score=np.zeros(len(pairs))
for i,(a,b) in enumerate(pairs):
 ta,tb=mtf[a],mtf[b];test[i]=bool((ta|tb)&query)
 for x in ta:
  for y in tb:score[i]=max(score[i],tfscore.get((min(x,y),max(x,y)),0))
pd.DataFrame({'motif_a':[a for a,b in pairs],'motif_b':[b for a,b in pairs],'chip_atlas_max_colo_score':score,'chip_testable':test}).to_csv(OUT/'chip_atlas_motif_pair_scores.tsv',sep='\t',index=False)
# prediction scores same definitions as prior STRING/MS analyses
scores={'causal':{},'spearman':{}}
for lin in LINS:
 a=np.zeros(len(pairs));d=raw[raw.cell_type.eq(lin)]
 for x,y,v in d[['motif_a','motif_b','score']].itertuples(index=False):a[pidx[(x,y)]]=abs(v)
 scores['causal'][lin]=a
head=pd.read_csv(GENE/'hsc_mpp.gene_by_motif.csv',nrows=1).columns;motifs=[x for x in head if x!='Accessibility' and not x.startswith('Unnamed:')];mi={m:i for i,m in enumerate(motifs)};ii=np.array([mi[a] for a,b in pairs]);jj=np.array([mi[b] for a,b in pairs])
for lin in LINS:
 x=pd.read_csv(GENE/f'{lin}.gene_by_motif.csv')[motifs].to_numpy(dtype=np.float32,copy=False);co=spearmanr(x,axis=0).statistic;scores['spearman'][lin]=np.abs(np.nan_to_num(co[ii,jj],nan=0))
rows=[]
for cut in [4,9]:
 pos=score>=cut;uni=np.where(test)[0];bn=len(uni);bp=int(pos[uni].sum())
 for method,label in [('causal','LiNGAM'),('spearman','Spearman')]:
  for cv in COV:
   k=int(np.ceil(len(pairs)*cv));sel=np.zeros((len(pairs),4),bool)
   for j,l in enumerate(LINS):sel[np.argsort(-scores[method][l],kind='mergesort')[:k],j]=True
   nl=sel.sum(1)
   for cl,mask in [('common_4of4',nl==4),('specific_1of4',nl==1),('common_plus_specific',(nl==4)|(nl==1))]:
    ix=np.where(mask&test)[0];n=len(ix);p=int(pos[ix].sum());lo,hi=proportion_confint(p,n,method='wilson') if n else (np.nan,np.nan);od,pv=fisher_exact([[p,n-p],[bp,bn-bp]],alternative='greater') if n else (np.nan,np.nan)
    rows.append({'chip_atlas_class':klass,'colo_cutoff':cut,'method':label,'coverage_pct':cv*100,'edge_class':cl,'raw_edge_n':int(mask.sum()),'chip_testable_n':n,'chip_supported_n':p,'support_precision':p/n if n else np.nan,'precision_ci_low':lo,'precision_ci_high':hi,'universe_n':bn,'universe_supported_n':bp,'universe_precision':bp/bn,'fisher_or':od,'fisher_p_greater':pv})
pd.DataFrame(rows).to_csv(OUT/'predicted_edges_chip_atlas_enrichment.csv',index=False)
pd.DataFrame({'metric':['mapped_model_tfs','downloaded_query_tfs','mapped_tf_pair_scores','motif_pairs','chip_testable_motif_pairs'],'n':[len(t2m),len(query),len(tfscore),len(pairs),int(test.sum())]}).to_csv(OUT/'mapping_qc.csv',index=False)

## BioID and AP-MS

Union of BioID and AP-MS non-self bait/prey pairs, with uppercase symbols and unordered pair deduplication. Fisher columns retain the historical overlapping background.

In [ ]:
#!/usr/bin/env python3
from pathlib import Path
import numpy as np,pandas as pd
from scipy.stats import spearmanr,fisher_exact
from statsmodels.stats.proportion import proportion_confint
OUT=OUTPUT_ROOT/'bioid_apms'
OUT.mkdir(parents=True,exist_ok=True)
XLSX=MS_XLSX
LINS=['hsc_mpp','memp','elp','cmp'];COV=[.01,.02,.05,.10]
mp=pd.read_csv(MAPPING,encoding='utf-8-sig').dropna();mp.tf=mp.tf.astype(str).str.upper();mtf=mp.groupby('motif').tf.agg(lambda x:set(x)).to_dict();t2m=mp.groupby('tf').motif.agg(lambda x:set(x)).to_dict()
def sheet(n):
 d=pd.read_excel(XLSX,sheet_name=n,header=9)[['Bait','PreyGene']].dropna();d.Bait=d.Bait.astype(str).str.upper();d.PreyGene=d.PreyGene.astype(str).str.upper();return d[d.Bait.ne(d.PreyGene)]
bio,ap=sheet('Supplementary Data 1B BioID'),sheet('Supplementary Data 1C AP-MS');bait=set(bio.Bait)|set(ap.Bait);pos_tf=set()
for d in [bio,ap]:
 for a,b in d[['Bait','PreyGene']].itertuples(index=False):pos_tf.add(tuple(sorted((a,b))))
raw=pd.read_csv(UNDIRECTED);raw['motif_a']=raw[['motif1','motif2']].min(axis=1);raw['motif_b']=raw[['motif1','motif2']].max(axis=1);pairs=sorted(set(zip(raw.motif_a,raw.motif_b)));assert len(pairs)==21120;pidx={p:i for i,p in enumerate(pairs)}
testable=np.zeros(len(pairs),bool);positive=np.zeros(len(pairs),bool)
for i,(a,b) in enumerate(pairs):
 ta,tb=mtf[a],mtf[b];testable[i]=bool((ta|tb)&bait);positive[i]=any(tuple(sorted((x,y))) in pos_tf for x in ta for y in tb if x!=y)
uni=np.where(testable)[0];bg_n=len(uni);bg_pos=int(positive[uni].sum());pd.DataFrame({'motif_a':[pairs[i][0] for i in uni],'motif_b':[pairs[i][1] for i in uni],'massspec_supported':positive[uni]}).to_csv(OUT/'massspec_testable_motif_pair_universe.tsv',sep='\t',index=False)
scores={'causal':{},'spearman':{}}
for lin in LINS:
 a=np.zeros(len(pairs));d=raw[raw.cell_type.eq(lin)]
 for x,y,v in d[['motif_a','motif_b','score']].itertuples(index=False):a[pidx[(x,y)]]=abs(v)
 scores['causal'][lin]=a
head=pd.read_csv(GENE/'hsc_mpp.gene_by_motif.csv',nrows=1).columns;motifs=[x for x in head if x!='Accessibility' and not x.startswith('Unnamed:')];mi={m:i for i,m in enumerate(motifs)};ii=np.array([mi[a] for a,b in pairs]);jj=np.array([mi[b] for a,b in pairs])
for lin in LINS:
 x=pd.read_csv(GENE/f'{lin}.gene_by_motif.csv')[motifs].to_numpy(dtype=np.float32,copy=False);co=spearmanr(x,axis=0).statistic;scores['spearman'][lin]=np.abs(np.nan_to_num(co[ii,jj],nan=0))
rows=[]
for method,label in [('causal','LiNGAM'),('spearman','Spearman')]:
 for cov in COV:
  k=int(np.ceil(len(pairs)*cov));sel=np.zeros((len(pairs),len(LINS)),dtype=bool)
  for j,l in enumerate(LINS): sel[np.argsort(-scores[method][l],kind='mergesort')[:k],j]=True
  nl=sel.sum(1)
  for cls,mask in [('common_4of4',nl==4),('specific_1of4',nl==1),('common_plus_specific',(nl==4)|(nl==1))]:
   ind=np.where(mask&testable)[0];n=len(ind);p=int(positive[ind].sum());lo,hi=proportion_confint(p,n,alpha=.05,method='wilson') if n else (np.nan,np.nan);od,pv=fisher_exact([[p,n-p],[bg_pos,bg_n-bg_pos]],alternative='greater') if n else (np.nan,np.nan)
   rows.append({'method':label,'coverage_pct':cov*100,'top_n_per_lineage':k,'edge_class':cls,'raw_edge_n':int(mask.sum()),'massspec_testable_n':n,'massspec_supported_n':p,'support_precision':p/n if n else np.nan,'precision_ci_low':lo,'precision_ci_high':hi,'universe_n':bg_n,'universe_supported_n':bg_pos,'universe_precision':bg_pos/bg_n,'fold_over_universe':(p/n)/(bg_pos/bg_n) if n else np.nan,'fisher_or':od,'fisher_p_greater':pv})
pd.DataFrame(rows).to_csv(OUT/'predicted_edges_massspec_enrichment.csv',index=False)
pd.DataFrame({'metric':['mapped_model_TFs','MS_bait_TFs','MS_union_positive_TF_pairs','causal_union_motif_pairs','MS_testable_motif_pairs','MS_supported_motif_pairs'],'n':[len(t2m),len(bait),len(pos_tf),len(pairs),bg_n,bg_pos]}).to_csv(OUT/'predicted_edges_massspec_mapping_qc.csv',index=False)

## Source provenance

Adapted from the following analysis scripts. SHA-256 values identify the original local source snapshots, before path portability changes. Source data are not bundled.

| Source script | SHA-256 |
| --- | --- |
| `validate_hspc_motif_networks_string_v11_5.py` | `ae560ed298f7a00cfaa2399d42ea6c4bcf6fd8de6840472d34b306b7be321ac5` |
| `benchmark_chip_atlas_class_enrichment.py` | `c7577cc74b3af799965f1789177d2d73d37e5c5f72ab08c71b82bc93029f092e` |
| `benchmark_predicted_edges_massspec_enrichment.py` | `37ff36be843ccea77ac451c983aee20e9b0fe5bb66a45dd97581b1c6b3dc9cd3` |